In [1]:
import numpy as np
import pandas as pd

from datetime import datetime, timezone, timedelta


In [2]:
import hax
pax_version = '6.10.1'

hax.init(experiment='XENON1T',
         pax_version_policy=pax_version,
         minitree_paths=['scratch-midway2/miniforest/pax_v'+ pax_version,
                         '/project2/lgrandi/xenon1t/minitrees/pax_v'+ pax_version,
                         '/dali/lgrandi/xenon1t/minitrees/pax_v'+ pax_version,
                        ],
         main_data_paths=['/dali/lgrandi/xenon1t/processed/pax_v'+ pax_version,
                          '/project2/lgrandi/xenon1t/processed/pax_v'+ pax_version,
                         ],)

dsets = hax.runs.datasets

dsets = hax.runs.tags_selection(dsets, include=['sciencerun0','sciencerun1','GW'],
                            exclude=['NoMV','Flash','PMTramping','readoutbug','flash','bad', 'messy', '*trip', '*quake','test','NG','MVoff'])

dsets = dsets[
    (dsets['location'] != "")
    & (dsets.source__type == 'none')
]



In [3]:
dataset_index = []
dataset_start = []
dataset_end = []
for i in range(0, np.shape(dsets)[0]):
    id = dsets.index[i]
    dataset_index.append(i)
    dataset_start.append(dsets.start[id].replace(tzinfo=timezone.utc).timestamp())
    dataset_end.append(dsets.end[id].replace(tzinfo=timezone.utc).timestamp())


In [4]:
# intialise data of lists.

from datetime import datetime, timezone, timedelta

GW_8_23_datetime = datetime(2017, 8, 23, 13, 13, 58, tzinfo=timezone.utc)
GW_8_18_datetime = datetime(2017, 8, 18, 2, 25, 9, tzinfo=timezone.utc)
GW_8_17_datetime = datetime(2017, 8, 17, 12, 41, 4, tzinfo=timezone.utc)
GW_7_29_datetime = datetime(2017, 7, 29, 18, 56, 29, tzinfo=timezone.utc)
GW_1_04_datetime = datetime(2017, 1, 4, 10, 11, 58, tzinfo=timezone.utc)

GW_1_04_time = GW_1_04_datetime.timestamp()
GW_7_29_time = GW_7_29_datetime.timestamp()
GW_8_17_time = GW_8_17_datetime.timestamp()
GW_8_18_time = GW_8_18_datetime.timestamp()
GW_8_23_time = GW_8_23_datetime.timestamp()

GW = ['GW170104', 'GW170729', 'GW170817', 'GW170818', 'GW170823']

data = {'Event': GW,
        'Event_time': [GW_1_04_time, GW_7_29_time, GW_8_17_time, GW_8_18_time, GW_8_23_time],
        }
Data = pd.DataFrame(data)

Data

,Event,Event_time
0,GW170104,1.483525e+09
1,GW170729,1.501355e+09
2,GW170817,1.502974e+09
3,GW170818,1.503023e+09
4,GW170823,1.503494e+09


In [5]:
Data['Time before'] = ''
Data['Time after'] = ''
Data['dsets_index'] = ''

for j in range(0, np.shape(Data['Event_time'])[0]):
    for i in range(0, np.size(dataset_start)):
        if Data['Event_time'][j] >= dataset_start[i] and Data['Event_time'][j] <= dataset_end[i]:
            Data['Time before'][j] = Data['Event_time'][j] - dataset_start[i] 
            Data['Time after'][j] = dataset_end[i] - Data['Event_time'][j]
            Data['dsets_index'][j] = dataset_index[i]



/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing

In [6]:
Data

,Event,Event_time,Time before,Time after,dsets_index
0,GW170104,1.483525e+09,2336,1268,595
1,GW170729,1.501355e+09,2272,1332,3481
2,GW170817,1.502974e+09,1982,1622,3540
3,GW170818,1.503023e+09,848,2756,3554
4,GW170823,1.503494e+09,1653,1951,3673


Datasets are completely within 500 seconds of GW

In [10]:
def extract_data(datasets):
    exclude_runs = []
    #exclude_runs = [16087, 15703, 15798, 16020, 15722, 15954, 16188]
    datasets = datasets.query('not number in @exclude_runs')
    data = hax.minitrees.load(datasets.number.values, ["Basics",
                                                "Fundamentals",
                                                "Proximity",
                                                "TotalProperties",
                                                "FlashIdentification",
                                                "PeaksBeforeTrigger",
                                                "LoneSignals",
                                                "TailCut",
                                               ], 
                          #preselection=["abs(nearest_muon_veto_trigger)<2e6"],
    #                           num_workers=1,
                          #force_reload=True
                             )
    return data

In [9]:
#comment out TailCut, LoneSignals and PeaksBeforeTrigger before running this
GW170104_data = extract_data(dsets[dsets.index == dsets.index[595]]) 

/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/dask/base.py:835: UserWarning: The get= keyword has been deprecated. Please use the scheduler= keyword instead with the name of the desired scheduler like 'threads' or 'processes'
  warnings.warn("The get= keyword has been deprecated. "


In [11]:
GW170729_data = extract_data(dsets[dsets.index == dsets.index[3481]])
GW170817_data = extract_data(dsets[dsets.index == dsets.index[3540]])
GW170818_data = extract_data(dsets[dsets.index == dsets.index[3554]])
GW170823_data = extract_data(dsets[dsets.index == dsets.index[3673]])

In [48]:
def total_events(df, GW_time):
    total_events = 0
    total_event_time = 0
    for i in range (0, np.shape(df)[0]):
        if abs(df['event_time'][i] * 1e-9 - GW_time) < 500:
            total_events = total_events + 1  
            total_event_time = total_event_time + df['event_duration'][i]*1e-9 #in seconds
    return total_events, total_event_time

In [49]:
total_events(GW170104_data, Data['Event_time'][0])

(5401, 14.075006000000114)

In [50]:
Data['total_events'] = ''
Data['total_event_time'] = ''
Data['total_events'][0], Data['total_event_time'][0] = total_events(GW170104_data, Data['Event_time'][0])
Data['total_events'][1], Data['total_event_time'][1] = total_events(GW170729_data, Data['Event_time'][1])
Data['total_events'][2], Data['total_event_time'][2] = total_events(GW170817_data, Data['Event_time'][2])
Data['total_events'][3], Data['total_event_time'][3] = total_events(GW170818_data, Data['Event_time'][3])
Data['total_events'][4], Data['total_event_time'][4] = total_events(GW170823_data, Data['Event_time'][4])
Data

/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  app.launch_new_instance()
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/st

,Event,Event_time,Time before,Time after,dsets_index,total_events,s2only_live_events,s2only_livetime,total_event_time
0,GW170104,1.483525e+09,2336,1268,595,5401,0,0,14.075
1,GW170729,1.501355e+09,2272,1332,3481,5324,5224,979.255,13.909
2,GW170817,1.502974e+09,1982,1622,3540,5326,5300,993.128,13.8575
3,GW170818,1.503023e+09,848,2756,3554,5371,5329,990.196,13.9313
4,GW170823,1.503494e+09,1653,1951,3673,5326,5298,992.753,13.836


In [52]:
Table = Data[['Event', 'total_events', 'total_event_time']].copy()
Table

,Event,total_events,total_event_time
0,GW170104,5401,14.075
1,GW170729,5324,13.909
2,GW170817,5326,13.8575
3,GW170818,5371,13.9313
4,GW170823,5326,13.836


## DAQVeto

Adopted from https://github.com/XENON1T/lax/blob/v1.0.0/lax/lichens/sciencerun0.py#L103

### EndOfRunCheck
Removes the last 21 seconds of each run.
From above table we can see that it is not an issue.

### HEVCheck

High energy veto is used only in calibration modes and goes TRUE when a very large S2 is present in the detector.

In [14]:
def HEVCheck(df):
    for i in range (0, np.shape(df)[0]):
        if (abs(df['nearest_hev'][i]) < df['event_duration'][i] / 2):
            print("problem")
            

In [15]:
HEVCheck(GW170104_data)

In [16]:
HEVCheck(GW170729_data)

In [17]:
HEVCheck(GW170817_data)

In [18]:
HEVCheck(GW170818_data)

In [19]:
HEVCheck(GW170823_data)

Verified that we don't have deadtime due to this

### BusyCheck

Busy veto goes TRUE if any digitizer nearly fills its memory buffer.

if (abs(df['nearest_busy'][i]) > df['event_duration'][i] / 2): 

### BusyTypeCheck

The BusyTypeCheck ensures that the most recent signal of this type is “*_OFF”, meaning that the event is not happening within a busy.

if (df['previous_busy_off'][i] < df['previous_busy_on'][i]):

## Muon Veto

Adopted from https://github.com/XENON1T/lax/blob/master/lax/lichens/sciencerun0.py#L896

The events when MV was not working are removed. nearest_muon_veto_trigger should be within 20 seconds of an event.

In [26]:
def MVnotWorking(df):
    for i in range (0, np.shape(df)[0]):
        if (df['nearest_muon_veto_trigger'][i] < -2e10 and df['nearest_muon_veto_trigger'][i] > 2e10):
            print("problem")

In [27]:
MVnotWorking(GW170823_data)

In [28]:
MVnotWorking(GW170818_data)

In [29]:
MVnotWorking(GW170817_data)

In [30]:
MVnotWorking(GW170729_data)

In [31]:
MVnotWorking(GW170104_data)

It means that Muon Veto is working for our dataset.

Events are excluded if the closest Muon Veto trigger falls in a [-2ms,+3ms] time window with respect to the reference point inside the waveform.

if (df['nearest_muon_veto_trigger'][i] > -2e6 and df['nearest_muon_veto_trigger'][i] < 3e6):

## Flash

Any event within a flash?

In [32]:
def InFlash(df):
    for i in range (0, np.shape(df)[0]):
        if (df['inside_flash'][i] == True):
            print("problem")


In [33]:
InFlash(GW170104_data)
InFlash(GW170729_data)
InFlash(GW170817_data)
InFlash(GW170818_data)
InFlash(GW170823_data)

Any event close to the flash?

In [34]:
def nearFlash(df):
    for i in range (0, np.shape(df)[0]):
        if (df['nearest_flash'][i] < 120e9):
            print("problem")

In [35]:
nearFlash(GW170104_data)
nearFlash(GW170729_data)
nearFlash(GW170817_data)
nearFlash(GW170818_data)
nearFlash(GW170823_data)

Any event just before the flash?

In [36]:
def beforeFlash(df):
    for i in range (0, np.shape(df)[0]):
        if (df['nearest_flash'][i] < (-10e9 - df['flashing_width'][i] * 1e9)):
            print("problem")

In [37]:
beforeFlash(GW170104_data)
beforeFlash(GW170729_data)
beforeFlash(GW170817_data)
beforeFlash(GW170818_data)
beforeFlash(GW170823_data)

Since we excluded runs with "flash" tag that's why probably there is no "bad" event. The fact that we have NaN values in the desired variable further indicates that.

In [38]:
GW170817_data['nearest_flash']

0       NaN
1       NaN
2       NaN
3       NaN
4       NaN
5       NaN
6       NaN
7       NaN
8       NaN
9       NaN
10      NaN
11      NaN
12      NaN
13      NaN
14      NaN
15      NaN
16      NaN
17      NaN
18      NaN
19      NaN
20      NaN
21      NaN
22      NaN
23      NaN
24      NaN
25      NaN
26      NaN
27      NaN
28      NaN
29      NaN
         ..
18743   NaN
18744   NaN
18745   NaN
18746   NaN
18747   NaN
18748   NaN
18749   NaN
18750   NaN
18751   NaN
18752   NaN
18753   NaN
18754   NaN
18755   NaN
18756   NaN
18757   NaN
18758   NaN
18759   NaN
18760   NaN
18761   NaN
18762   NaN
18763   NaN
18764   NaN
18765   NaN
18766   NaN
18767   NaN
18768   NaN
18769   NaN
18770   NaN
18771   NaN
18772   NaN
Name: nearest_flash, Length: 18773, dtype: float64

## s2Tails

Cut on tails after big S2s to reduce the contribution of AC events.

We divide s2_area for 50 previous s2s with the time passed after them and take the maximum value. (the commented code below)
That maximum value is s2_over_tdiff which is compared with a predefined threshold.

In [39]:
#https://github.com/XENON1T/hax/blob/master/hax/treemakers/trigger.py

def s2_over_tdiff(df):
    look_back = 50
    s2 = df['s2'].values
    s2[np.isnan(s2)] = 0
    t = df['event_time'].values

    s2_over_tdiff_lookback = np.zeros((len(t), look_back + 1))

    for i in range(1, look_back + 1):
        s2_over_tdiff_lookback[i:, i] = s2[:-i]/(t[i:] - t[:-i])
    s2_over_tdiff = s2_over_tdiff_lookback.max(axis=1)
    return s2_over_tdiff

In [40]:
GW170104_data['s2_over_tdiff'] = s2_over_tdiff(GW170104_data)

## Lone S1s

The rate of Lone-S1 signal is uniform and their spectrum is the same on the time axis. 
The lone-S1 spectrum has a rate of [0.7, 1.1] Hz. (ton year paper)
Let take rate of 1 Hz. Deadtime cut per event is 2 milliseconds. So, 0.002 is lost and 0.998 is accepted.

## Clean Window

There is no single electron S2s before the trigger.
There are less than 2 S1s before the trigger.

## Pretrigger junk

Area of junk (mostly lone_hit) peaks before trigger happens is a good discriminator of Accidental Coincidence noise. Area before trigger window should be below 40PE for CEvNS event. It has an acceptance of 96%

# s2only

Dead time due to DAQVeto, FlashIdentification, MuonVeto, and lone S1s.

In [41]:
def s2only_livetime(df, GW_time):
    live_events = 0            
    for i in range (0, np.shape(df)[0]):
        if abs(df['event_time'][i] * 1e-9 - GW_time) < 500:
            if (abs(df['nearest_busy'][i]) > df['event_duration'][i] / 2): #DAQ busycheck
                if ~(df['previous_busy_on'][i] < 60e9) | (df['previous_busy_off'][i] < df['previous_busy_on'][i]): #DAQ busytypecheck
                    if (df['nearest_muon_veto_trigger'][i] < -2e6 or df['nearest_muon_veto_trigger'][i] > 3e6): #MuonVeto
                        live_events = live_events + 1            
    return live_events  

In [42]:
Data['s2only_live_events'] = ''
Data['s2only_live_events'][0] = 0 #s2only is not analyzed for SR0
Data['s2only_live_events'][1] = s2only_livetime(GW170729_data, Data['Event_time'][1])
Data['s2only_live_events'][2] = s2only_livetime(GW170817_data, Data['Event_time'][2])
Data['s2only_live_events'][3] = s2only_livetime(GW170818_data, Data['Event_time'][3])
Data['s2only_live_events'][4] = s2only_livetime(GW170823_data, Data['Event_time'][4])
Data

/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  from ipykernel import kernelapp as app
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  app.launch_new_instance()
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation

,Event,Event_time,Time before,Time after,dsets_index,total_events,s2only_live_events
0,GW170104,1.483525e+09,2336,1268,595,5401,0
1,GW170729,1.501355e+09,2272,1332,3481,5324,5224
2,GW170817,1.502974e+09,1982,1622,3540,5326,5300
3,GW170818,1.503023e+09,848,2756,3554,5371,5329
4,GW170823,1.503494e+09,1653,1951,3673,5326,5298


In [67]:
s2only_acceptance = Data['s2only_live_events']/Data['total_events']
Data['s2only_livetime'] = (1000-Data['total_event_time'])*s2only_acceptance*0.998 
#2 seconds for Lone-S1 signal
Data

,Event,Event_time,Time before,Time after,dsets_index,total_events,s2only_live_events,s2only_livetime,total_event_time,NR_live_events,NR_livetime,CEvNS_live_events,CEvNS_livetime,CEvNS_live_events_till_s2Tail
0,GW170104,1.483525e+09,2336,1268,595,5401,0,0,14.075,5164,956.119,0,0,0
1,GW170729,1.501355e+09,2272,1332,3481,5324,5224,965.634,13.909,5002,939.519,4240,796.394,4469
2,GW170817,1.502974e+09,1982,1622,3540,5326,5300,979.366,13.8575,5077,953.248,4265,800.789,4513
3,GW170818,1.503023e+09,848,2756,3554,5371,5329,976.401,13.9313,5101,949.73,4319,804.133,4570
4,GW170823,1.503494e+09,1653,1951,3673,5326,5298,979.018,13.836,5056,949.305,4251,798.16,4478


# Nuclear Recoil

Dead time due to DAQVeto, FlashIdentification, MuonVeto, and s2Tails.

In [53]:
def NR_live_events(df, GW_time):
    live_events = 0            
    for i in range (0, np.shape(df)[0]):
        if abs(df['event_time'][i] * 1e-9 - GW_time) < 500:
            if (abs(df['nearest_busy'][i]) > df['event_duration'][i] / 2): #DAQ busycheck
                if ~(df['previous_busy_on'][i] < 60e9) | (df['previous_busy_off'][i] < df['previous_busy_on'][i]): #DAQ busytypecheck
                    if (df['nearest_muon_veto_trigger'][i] < -2e6 or df['nearest_muon_veto_trigger'][i] > 3e6): #MuonVeto
                        if ((~(df['s2_over_tdiff'][i] >= 0)) | (df['s2_over_tdiff'][i] < 0.04)): #s2Tail Cut 
                            live_events = live_events + 1            
    return live_events  

In [54]:
Data['NR_live_events'] = ''
Data['NR_live_events'][0] = NR_live_events(GW170104_data, Data['Event_time'][0])
Data['NR_live_events'][1] = NR_live_events(GW170729_data, Data['Event_time'][1])
Data['NR_live_events'][2] = NR_live_events(GW170817_data, Data['Event_time'][2])
Data['NR_live_events'][3] = NR_live_events(GW170818_data, Data['Event_time'][3])
Data['NR_live_events'][4] = NR_live_events(GW170823_data, Data['Event_time'][4])
Data

/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  from ipykernel import kernelapp as app
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  app.launch_new_instance()
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation

,Event,Event_time,Time before,Time after,dsets_index,total_events,s2only_live_events,s2only_livetime,total_event_time,NR_live_events
0,GW170104,1.483525e+09,2336,1268,595,5401,0,0,14.075,5164
1,GW170729,1.501355e+09,2272,1332,3481,5324,5224,979.255,13.909,5002
2,GW170817,1.502974e+09,1982,1622,3540,5326,5300,993.128,13.8575,5077
3,GW170818,1.503023e+09,848,2756,3554,5371,5329,990.196,13.9313,5101
4,GW170823,1.503494e+09,1653,1951,3673,5326,5298,992.753,13.836,5056


In [68]:
NR_acceptance = Data['NR_live_events']/Data['total_events']
Data['NR_livetime'] = (1000-Data['total_event_time'])*NR_acceptance
Data

,Event,Event_time,Time before,Time after,dsets_index,total_events,s2only_live_events,s2only_livetime,total_event_time,NR_live_events,NR_livetime,CEvNS_live_events,CEvNS_livetime,CEvNS_live_events_till_s2Tail
0,GW170104,1.483525e+09,2336,1268,595,5401,0,0,14.075,5164,942.662,0,0,0
1,GW170729,1.501355e+09,2272,1332,3481,5324,5224,965.634,13.909,5002,926.451,4240,796.394,4469
2,GW170817,1.502974e+09,1982,1622,3540,5326,5300,979.366,13.8575,5077,940.039,4265,800.789,4513
3,GW170818,1.503023e+09,848,2756,3554,5371,5329,976.401,13.9313,5101,936.499,4319,804.133,4570
4,GW170823,1.503494e+09,1653,1951,3673,5326,5298,979.018,13.836,5056,936.171,4251,798.16,4478


# Electronic Recoil

https://xe1t-wiki.lngs.infn.it/doku.php?id=xenon:xenon1:low_energy_er:sr1_take2:results

Start with the SR1 runlist (248 days). Remove all runs within 24 hours of Kr83m calibration to remove left-over Kr83m contamination (228 days), and the same for Rn220 (226.93 days). 

So livetime will be same as nuclear recoil as long as that run is in the list.

In [5]:
#https://github.com/XENON1T/LowER/blob/master/LowER/data/runlists/sr1_nokr83m_norn220.npy
runlist = "sr1_nokr83m_norn220.npy"
runs = np.load(runlist)


In [8]:
dsets[dsets.index == dsets.index[3481]]['number']

12101    11661
Name: number, dtype: int64

In [13]:
dsets[dsets.index == dsets.index[3540]]['number'] 

12180    12185
Name: number, dtype: int64

In [14]:
dsets[dsets.index == dsets.index[3554]]['number'] 

12194    12199
Name: number, dtype: int64

In [15]:
dsets[dsets.index == dsets.index[3673]]['number'] 

12344    12349
Name: number, dtype: int64

In [9]:
11661 in runs

True

In [16]:
12185 in runs

True

In [17]:
12199 in runs

True

In [18]:
12349 in runs

True

In [72]:
Data['ER_livetime'] = Data['NR_livetime']
Data['ER_livetime'][0] = 0
Data

/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  from ipykernel import kernelapp as app


,Event,Event_time,Time before,Time after,dsets_index,total_events,s2only_live_events,s2only_livetime,total_event_time,NR_live_events,NR_livetime,CEvNS_live_events,CEvNS_livetime,CEvNS_live_events_till_s2Tail,ER_livetime
0,GW170104,1.483525e+09,2336,1268,595,5401,0,0,14.075,5164,942.662,0,0,0,0
1,GW170729,1.501355e+09,2272,1332,3481,5324,5224,965.634,13.909,5002,926.451,4240,785.317,4469,926.451
2,GW170817,1.502974e+09,1982,1622,3540,5326,5300,979.366,13.8575,5077,940.039,4265,789.692,4513,940.039
3,GW170818,1.503023e+09,848,2756,3554,5371,5329,976.401,13.9313,5101,936.499,4319,792.931,4570,936.499
4,GW170823,1.503494e+09,1653,1951,3673,5326,5298,979.018,13.836,5056,936.171,4251,787.117,4478,936.171


# CEvNS

Dead time due to DAQVeto, FlashIdentification, MuonVeto, s2Tails and CleanWindow. 

In [56]:
def CEvNS_live_events(df, GW_time):
    live_events = 0            
    for i in range (0, np.shape(df)[0]):
        if abs(df['event_time'][i] * 1e-9 - GW_time) < 500:
            if (abs(df['nearest_busy'][i]) > df['event_duration'][i] / 2): #DAQ busycheck
                if ~(df['previous_busy_on'][i] < 60e9) | (df['previous_busy_off'][i] < df['previous_busy_on'][i]): #DAQ busytypecheck
                    if (df['nearest_muon_veto_trigger'][i] < -2e6 or df['nearest_muon_veto_trigger'][i] > 3e6): #MuonVeto
                        if ((~(df['s2_over_tdiff'][i] >= 0)) | (df['s2_over_tdiff'][i] < 0.012)): #s2Tail Cut 
                            #if df['area_before_main_s2'][i] - df['s1'][i] < 40: #PreTriggerJunk Cut
                                if ((df['n_s2_before_trigger'][i] < 1) & (df['n_s1_before_trigger'][i] <= 1)): #CleanWindow
                                    live_events = live_events + 1            
    return live_events

In [57]:
Data['CEvNS_live_events'] = ''
Data['CEvNS_live_events'][0] = 0 #CEvNS is not analyzed for SR0
Data['CEvNS_live_events'][1] = CEvNS_live_events(GW170729_data, Data['Event_time'][1])
Data['CEvNS_live_events'][2] = CEvNS_live_events(GW170817_data, Data['Event_time'][2])
Data['CEvNS_live_events'][3] = CEvNS_live_events(GW170818_data, Data['Event_time'][3])
Data['CEvNS_live_events'][4] = CEvNS_live_events(GW170823_data, Data['Event_time'][4])
Data

/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  from ipykernel import kernelapp as app
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  app.launch_new_instance()
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation

,Event,Event_time,Time before,Time after,dsets_index,total_events,s2only_live_events,s2only_livetime,total_event_time,NR_live_events,NR_livetime,CEvNS_live_events
0,GW170104,1.483525e+09,2336,1268,595,5401,0,0,14.075,5164,956.119,0
1,GW170729,1.501355e+09,2272,1332,3481,5324,5224,979.255,13.909,5002,939.519,4240
2,GW170817,1.502974e+09,1982,1622,3540,5326,5300,993.128,13.8575,5077,953.248,4265
3,GW170818,1.503023e+09,848,2756,3554,5371,5329,990.196,13.9313,5101,949.73,4319
4,GW170823,1.503494e+09,1653,1951,3673,5326,5298,992.753,13.836,5056,949.305,4251


In [69]:
CEvNS_acceptance = Data['CEvNS_live_events']/Data['total_events']
Data['CEvNS_livetime'] = (1000-Data['total_event_time'])*CEvNS_acceptance
Data

,Event,Event_time,Time before,Time after,dsets_index,total_events,s2only_live_events,s2only_livetime,total_event_time,NR_live_events,NR_livetime,CEvNS_live_events,CEvNS_livetime,CEvNS_live_events_till_s2Tail
0,GW170104,1.483525e+09,2336,1268,595,5401,0,0,14.075,5164,942.662,0,0,0
1,GW170729,1.501355e+09,2272,1332,3481,5324,5224,965.634,13.909,5002,926.451,4240,785.317,4469
2,GW170817,1.502974e+09,1982,1622,3540,5326,5300,979.366,13.8575,5077,940.039,4265,789.692,4513
3,GW170818,1.503023e+09,848,2756,3554,5371,5329,976.401,13.9313,5101,936.499,4319,792.931,4570
4,GW170823,1.503494e+09,1653,1951,3673,5326,5298,979.018,13.836,5056,936.171,4251,787.117,4478


Original Analysis has used PreTriggerJunk but I am unable to properly implement it. Combined total acceptance of PreTriggerJunk and CleanWindow is greater than 0.96 * 0.98 = 0.94  

In [59]:
def CEvNS_live_events(df, GW_time):
    live_events = 0            
    for i in range (0, np.shape(df)[0]):
        if abs(df['event_time'][i] * 1e-9 - GW_time) < 500:
            if (abs(df['nearest_busy'][i]) > df['event_duration'][i] / 2): #DAQ busycheck
                if ~(df['previous_busy_on'][i] < 60e9) | (df['previous_busy_off'][i] < df['previous_busy_on'][i]): #DAQ busytypecheck
                    if (df['nearest_muon_veto_trigger'][i] < -2e6 or df['nearest_muon_veto_trigger'][i] > 3e6): #MuonVeto
                        if ((~(df['s2_over_tdiff'][i] >= 0)) | (df['s2_over_tdiff'][i] < 0.012)): #s2Tail Cut 
                            #if df['area_before_main_s2'][i] - df['s1'][i] < 40: #PreTriggerJunk Cut
                                #if ((df['n_s2_before_trigger'][i] < 1) & (df['n_s1_before_trigger'][i] <= 1)): #CleanWindow
                                    live_events = live_events + 1            
    return live_events

In [60]:
Data['CEvNS_live_events_till_s2Tail'] = ''
Data['CEvNS_live_events_till_s2Tail'][0] = 0 #CEvNS is not analyzed for SR0
Data['CEvNS_live_events_till_s2Tail'][1] = CEvNS_live_events(GW170729_data, Data['Event_time'][1])
Data['CEvNS_live_events_till_s2Tail'][2] = CEvNS_live_events(GW170817_data, Data['Event_time'][2])
Data['CEvNS_live_events_till_s2Tail'][3] = CEvNS_live_events(GW170818_data, Data['Event_time'][3])
Data['CEvNS_live_events_till_s2Tail'][4] = CEvNS_live_events(GW170823_data, Data['Event_time'][4])
Data

/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  from ipykernel import kernelapp as app
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  app.launch_new_instance()
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation

,Event,Event_time,Time before,Time after,dsets_index,total_events,s2only_live_events,s2only_livetime,total_event_time,NR_live_events,NR_livetime,CEvNS_live_events,CEvNS_livetime,CEvNS_live_events_till_s2Tail
0,GW170104,1.483525e+09,2336,1268,595,5401,0,0,14.075,5164,956.119,0,0,0
1,GW170729,1.501355e+09,2272,1332,3481,5324,5224,979.255,13.909,5002,939.519,4240,796.394,4469
2,GW170817,1.502974e+09,1982,1622,3540,5326,5300,993.128,13.8575,5077,953.248,4265,800.789,4513
3,GW170818,1.503023e+09,848,2756,3554,5371,5329,990.196,13.9313,5101,949.73,4319,804.133,4570
4,GW170823,1.503494e+09,1653,1951,3673,5326,5298,992.753,13.836,5056,949.305,4251,798.16,4478


In [138]:
acceptance_of_CleanWindow = Data['CEvNS_live_events'][1:]/Data['CEvNS_live_events_till_s2Tail'][1:]
acceptance_of_CleanWindow

1    0.948758
2    0.945048
3    0.945077
4    0.949308
dtype: object

Thus by not having PreTriggerJunk, we are not too making big error. Our acceptance is greater implying greater calculated livetime than real values which means our analysis will be more stringent.

In [73]:
Table = Data[['Event', 's2only_livetime', 'NR_livetime', 'ER_livetime', 'CEvNS_livetime']].copy()
Table

,Event,s2only_livetime,NR_livetime,ER_livetime,CEvNS_livetime
0,GW170104,0,942.662,0,0
1,GW170729,965.634,926.451,926.451,785.317
2,GW170817,979.366,940.039,940.039,789.692
3,GW170818,976.401,936.499,936.499,792.931
4,GW170823,979.018,936.171,936.171,787.117


In [61]:
table = {'Event': GW,
        }
Table = pd.DataFrame(table)

Table

,Event
0,GW170104
1,GW170729
2,GW170817
3,GW170818
4,GW170823


In [65]:
Table['CEvNS_acceptance'] = CEvNS_acceptance
Table['NR_acceptance'] = NR_acceptance
Table['ER_acceptance'] = NR_acceptance
Table['s2only_acceptance'] = s2only_acceptance*0.998

In [66]:
Table

,Event,CEvNS_acceptance,NR_acceptance,ER_acceptance,s2only_acceptance
0,GW170104,0,0.956119,0.956119,0
1,GW170729,0.796394,0.939519,0.939519,0.979255
2,GW170817,0.800789,0.953248,0.953248,0.993128
3,GW170818,0.804133,0.94973,0.94973,0.990196
4,GW170823,0.79816,0.949305,0.949305,0.992753
